# Urdu Sukhan ASR v1: try it in Colab

Urdu speech to text, including speech that switches between Urdu and English, by [Loudness.ai](https://loudness.ai).
This notebook runs [Urdu Sukhan ASR v1](https://huggingface.co/Loudness-ai/urdu-sukhan-asr-v1) in your own Colab
session: record your voice, upload a file, or try public test clips.

**Before you start**

1. Request access on the [model page](https://huggingface.co/Loudness-ai/urdu-sukhan-asr-v1) and wait for approval.
   The model is free for non-commercial use under CC BY-NC 4.0; for commercial use,
   [contact us](https://loudness.ai/contact-us/).
2. Create a **Read** token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).
3. Optional, for speed: **Runtime → Change runtime type → T4 GPU**. On a GPU a clip takes a few seconds; on Colab's
   CPU, about half a minute.

Then run the cells from top to bottom.

**Your audio** stays in this Colab session, a Google machine under your own account. Nothing is sent to Loudness.ai.

In [ ]:
# 1. Install (about a minute)
!pip -q install faster-whisper==1.2.1 ctranslate2==4.8.3 av==15.1.0

In [ ]:
# 2. Log in to Hugging Face with your Read token (or add it to Colab's Secrets as HF_TOKEN)
from huggingface_hub import get_token, login

if not get_token():
    login()

In [ ]:
# 3. Download and load the model (CTranslate2 int8, about 820 MB)
from faster_whisper import WhisperModel
from huggingface_hub import snapshot_download
import numpy as np

path = snapshot_download("Loudness-ai/urdu-sukhan-asr-v1", allow_patterns=["ctranslate2/*"]) + "/ctranslate2"
try:
    model = WhisperModel(path, device="cuda", compute_type="int8_float16")
    list(model.transcribe(np.zeros(16000, dtype=np.float32), language="ur")[0])  # checks the GPU libraries
    print("Loaded on the GPU")
except Exception as error:
    print("No usable GPU, so the CPU is used:", str(error)[:120])
    model = WhisperModel(path, device="cpu", compute_type="int8", cpu_threads=2)

In [ ]:
# 4. The transcription settings we recommend
import time
from faster_whisper import decode_audio


def transcribe(audio, language="ur", beams=5):
    """language: "ur" (also keeps English words inside Urdu), "en", or None to detect it."""
    samples = decode_audio(audio, sampling_rate=16000) if isinstance(audio, str) else audio
    start = time.time()
    segments, info = model.transcribe(
        samples, language=language, task="transcribe", beam_size=beams,
        suppress_blank=False,              # lets the model write nothing on silence
        condition_on_previous_text=False,  # no errors carried from one 30-second window to the next
        without_timestamps=True)
    text = " ".join(s.text.strip() for s in segments).strip()
    print(f"[{len(samples) / 16000:.1f} s of audio in {time.time() - start:.1f} s]")
    return text or "(no speech)"

## Try public test clips

Clips from Common Voice, FLEURS, IndicVoices and UrduSpeech, as shown on the model card.

In [ ]:
from huggingface_hub import hf_hub_download

for name in ["02-urdu-english-2", "03-reply-alag-alag", "07-quiet-speech", "08-phone-line", "10-faint-hiss"]:
    clip = hf_hub_download("Loudness-ai/urdu-sukhan-asr-examples", f"examples/{name}.mp3", repo_type="dataset")
    print(name, "→", transcribe(clip), "\n")

## Record your voice

Run the cell, allow the microphone, speak, and wait for the time to run out.

In [ ]:
import base64
from IPython.display import Javascript, display
from google.colab import output

SECONDS = 10
display(Javascript('''
async function record(ms) {
  const stream = await navigator.mediaDevices.getUserMedia({audio: true});
  const recorder = new MediaRecorder(stream); const chunks = [];
  recorder.ondataavailable = e => chunks.push(e.data);
  const done = new Promise(r => recorder.onstop = r);
  recorder.start(); await new Promise(r => setTimeout(r, ms)); recorder.stop(); await done;
  stream.getTracks().forEach(t => t.stop());
  const buffer = await new Blob(chunks).arrayBuffer();
  let binary = ''; new Uint8Array(buffer).forEach(b => binary += String.fromCharCode(b));
  return btoa(binary);
}'''))
print(f"Recording for {SECONDS} seconds…")
data = output.eval_js(f"record({SECONDS * 1000})")
with open("recording.webm", "wb") as f:
    f.write(base64.b64decode(data))
print(transcribe("recording.webm"))

## Upload a file

WAV, MP3, M4A, OGG, FLAC or WebM. Long recordings are transcribed in 30-second windows.

In [ ]:
from google.colab import files

for name in files.upload():
    print(name, "→", transcribe(name))

## Names

Names are the model's weakest point. If you know which names to expect, pass them as a prompt:
`model.transcribe(samples, language="ur", initial_prompt="عمیر، عزیر، تانیا")`.

More: [technical report](https://loudness.ai/research/urdu-speech-recognition-whisper/) ·
[install guide](https://loudness.ai/developers/guides/install-urdu-sukhan-asr/) ·
[code](https://github.com/Loudness-ai/urdu-sukhan-asr)